In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Image Classification Data

The softmax regression of that section is ready to be trained. What
it lacks is data. Our examples will be images, and the choice of dataset
decides what our experiments can show. Which dataset should we use? For the
purposes of this book, a good one is hard enough that a better model scores
visibly higher, and small enough that a model trains in minutes inside a
notebook.

Unfortunately, the two best-known image datasets each meet only one of these
needs.
MNIST [@LeCun.Bottou.Bengio.ea.1998], 70,000 grayscale images of
handwritten digits with $28 \times 28$ pixels in 10 classes, is small but too
easy. Convolutional networks classify more than 99% of its test digits
correctly, and even classical methods reach about 97%, so models that differ
in kind differ in accuracy by a few points at most.
ImageNet [@Deng.Dong.Socher.ea.2009], with 1.2 million color images in
1,000 classes, separates models well, but training a network on it takes
hours to days of GPU time, a long wait for a notebook cell.

Fashion-MNIST [@Xiao.Rasul.Vollgraf.2017] was designed as a harder
replacement for MNIST. It keeps the format, 70,000 grayscale images of
$28 \times 28$ pixels in 10 classes, but shows garments instead of digits,
and garments are harder to tell apart: a shirt can look like a T-shirt, a
pullover, or a coat.
Linear classifiers reach about 83% accuracy on it and good convolutional
networks more than 90%, which leaves room to tell models apart. We use it for
the rest of this chapter and for much of the book.

In this section we load Fashion-MNIST in each framework, read it in
minibatches, check the shape of what the loader delivers, and look at a few
images before any model touches them. None of this is deep, but it is worth
doing with care, since every later classifier in the book reads its data
through the class we write here.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
import time
import torchvision
from torchvision import transforms

In [ ]:
d2l.use_svg_display()

## Loading the Dataset

Fashion-MNIST has 7,000 images of each of the ten garments named in
that section, from T-shirt to ankle boot. Its authors split them
into 60,000 training and 10,000 test images, 6,000 and 1,000 per class. Each
pixel stores a brightness between 0 and 255, with the background at 0.

As in that section, a `DataModule` holds the data and hands
minibatches to the `Trainer` through its methods `train_dataloader` and
`val_dataloader`. The class `FashionMNIST` downloads the dataset on first use
and keeps both splits in memory. It has two hyperparameters: `batch_size`,
the number of examples per minibatch, and `resize`, the height and width of
the images that the loader delivers. By default the images keep their native
$28 \times 28$ pixels; networks in later chapters that are built for larger
inputs ask for enlarged images instead.

The torchvision library downloads Fashion-MNIST and serves it as a dataset
object. Its `transform` argument resizes each image and converts it into a
tensor with values in $[0, 1]$.

In [ ]:
class FashionMNIST(d2l.DataModule):
    """The Fashion-MNIST dataset."""
    def __init__(self, batch_size=64, resize=(28, 28)):
        super().__init__()
        self.save_hyperparameters()
        trans = transforms.Compose([transforms.Resize(resize),
                                    transforms.ToTensor()])
        self.train = torchvision.datasets.FashionMNIST(
            root=self.root, train=True, transform=trans, download=True)
        self.val = torchvision.datasets.FashionMNIST(
            root=self.root, train=False, transform=trans, download=True)

We load the dataset and count the images in each split.

In [ ]:
data = FashionMNIST()
len(data.train), len(data.val)

There are only two splits: Fashion-MNIST ships no validation set.
In this book the 10,000 test images serve as validation data instead, and the
`Trainer` reports the loss on them after every epoch. This is convenient, and
it is not quite honest. A test set that we consult again and again while we
compare models stops being an unbiased test of the model we finally choose.
How much harm this does is a question that
that section answers with numbers.

The labels are class indices from 0 to 9. To read them, we map each index
to the name of its class.

In [ ]:
@d2l.add_to_class(FashionMNIST)
def text_labels(self, indices):
    """Return text labels."""
    labels = ['T-shirt', 'trouser', 'pullover', 'dress', 'coat',
              'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [labels[int(i)] for i in indices]

## Reading Minibatches

To a framework, an image is a three-dimensional tensor with $c$ *channels*,
one for each color, a height $h$, and a width $w$ in pixels. Here $c = 1$
because the images are grayscale; color photographs have $c = 3$. Frameworks
do not agree on where the channel axis goes: *channel-first* layout stores
an image as $c \times h \times w$, *channel-last* layout as
$h \times w \times c$.

The loader for the training split draws the examples in a new random order
in every epoch, so that the minibatches differ from epoch to epoch; the
loader for the validation split reads them in their stored order. Both
convert the pixel values from integers between 0 and 255 into floating-point
numbers between 0 and 1, and resize the images if asked to. Since 64 does
not divide 60,000, the last minibatch of an epoch is smaller than the others
(32 images here). Both loaders keep it, so that every epoch covers every
image.

`DataLoader` reads the dataset in minibatches. Its `num_workers` processes,
4 by default in `DataModule`, load and transform the next minibatches in
parallel while the model works on the current one.

In [ ]:
@d2l.add_to_class(FashionMNIST)
def get_dataloader(self, train):
    data = self.train if train else self.val
    return torch.utils.data.DataLoader(data, self.batch_size, shuffle=train,
                                       num_workers=self.num_workers)

Which layout does our loader use? We read one minibatch and look at the
shape of its first image.

In [ ]:
X, y = next(iter(data.train_dataloader()))
X[0].shape  # Channel-first: (channels, height, width)

A minibatch adds an axis for the examples in front, and its labels form a
vector with one entry per example.

In [ ]:
print(X.shape, X.dtype, y.shape, y.dtype)

The images arrive as 32-bit floats and the labels as integers. One minibatch
of images takes $64 \times 784 \times 4$ bytes, about 200 KB, and the whole
training set as floats about 190 MB. By today's standards this is tiny. The
dataset fits in memory many times over, so the loader never waits for a
disk; its time goes into converting, resizing, and collecting the images.

Is that work fast enough? A loader that is slower than the model leaves the
processor waiting for data. We time one pass over the training set without
any model.

In [ ]:
tic = time.time()
for X, y in data.train_dataloader():
    continue
f'{time.time() - tic:.2f} sec'

The number above holds only for the machine that ran this notebook, since
it depends on the number of loader workers, the framework, and the hardware.
Whatever its value, it is a lower bound: no epoch of training can take less
time than reading its data. When training seems slow, this is the first
number to check. If an epoch takes barely longer than the loader alone, the
model is waiting for data, and more loader workers or prefetching of the
next minibatches will help more than any change to the model.

## Inspecting the Images

We look at the data before we train on it. A look catches errors that no
loss curve reveals: labels attached to the wrong images, images flipped or
transposed, a channel axis in the wrong place. A model trains on such data
without complaint, only worse. The method `visualize` shows
the images of a minibatch, each titled with its label. It calls the library
function `d2l.show_images`, which lays out a list of images in a grid with
optional titles. Matplotlib would draw single-channel images in false
color, so `visualize` sets the gray color map. Since every image has a
single channel, reshaping removes the channel axis in either layout.

In [ ]:
@d2l.add_to_class(FashionMNIST)
def visualize(self, batch, nrows=1, ncols=8, labels=None):
    """Show the images of a batch in grayscale, titled by their labels."""
    X, y = batch
    if labels is None:
        labels = self.text_labels(y)
    X = d2l.reshape(X, (-1, *self.resize))  # Drop the channel axis
    d2l.show_images(X, nrows, ncols, titles=labels, cmap='gray')

We show three images of every class, one class per column: the first
three of each class in the validation set, which the validation loader
always delivers in the same order.

In [ ]:
X, y = next(iter(data.val_dataloader()))
labels = d2l.numpy(y)
idx = [int(np.flatnonzero(labels == c)[r]) for r in range(3) for c in range(10)]
titles = data.text_labels(range(10)) + [''] * 20
data.visualize([d2l.stack([X[i] for i in idx]), None], nrows=3, ncols=10,
               labels=titles)

The labels match the garments, the garments are upright, and the
background is dark, as stored. Note, however, how much some garments of
different classes look alike. The T-shirts, pullovers, coats, and shirts in
the columns above share an outline and differ in a collar, a button, or the
length of a sleeve, details that $28 \times 28$ pixels barely resolve; the
sandals, sneakers, and ankle boots form a second such family. A trouser or
a bag, by contrast, looks like nothing else. We can already guess where a
classifier will make most of its mistakes. How many it makes, and whether
our guess was right, we learn once we have trained one
(that section).

## Summary

Fashion-MNIST has 60,000 training and 10,000 test images of garments in 10
classes, grayscale with $28 \times 28$ pixels. It is small enough to train
on in minutes, and hard enough that better models score visibly higher,
unlike MNIST. The class `FashionMNIST` loads the published files, and its
loaders deliver images as floats in $[0, 1]$ and labels as class indices;
the training loader reshuffles every epoch, and both loaders keep the last,
incomplete minibatch. A minibatch of images is a four-dimensional tensor,
with one axis each for the examples, the channels, the height, and the
width. Before training, we look at the images with their labels, and we
time the loader, since no epoch can be faster than reading its data.

## Exercises

1. [code] **Class means.** Compute the mean image of each class over the
   training set, and show the ten means as a grid of images.
    1. Compute the Euclidean distance between every pair of class means and
       present the distances as a $10 \times 10$ table. Which pairs are
       closest, and which class is farthest from all others?
    1. Predict from the table which classes a linear classifier will confuse
       most often and which it will almost never confuse. Keep your
       prediction; that section computes the answer.
    1. Name a pair of classes that a person tells apart easily although
       their means are close, and say which pixels carry the difference.

1. **Shuffling and validation.** `get_dataloader` reshuffles the training
   split every epoch and reads the validation split in the same order every
   epoch. Suppose the validation loader, with the default `batch_size=64`,
   were reshuffled every epoch as well.
    1. Explain whether the validation loss and accuracy, averaged over all
       10,000 validation examples, would change.
    1. The `Trainer` weights each validation batch by its number of
       examples. Suppose it took the unweighted mean of the per-batch
       values instead. Explain whether that mean would change.
    1. Explain whether the images that `visualize` shows for the first
       validation batch would change.

1. [code] **Reshaping versus transposing.** `visualize` removes the
   channel axis with a reshape, which is correct only because $c = 1$.
    1. A minibatch of color images has shape $(n, 3, h, w)$ in channel-first
       layout. Show that reshaping it to $(n, h, w, 3)$ does not move the
       channel axis to the end, by computing which stored value lands at
       position $(0, 0, 0, 1)$ of the result.
    1. Write the transposition that converts it correctly, apply both
       versions to a minibatch of 8 random arrays of shape $(3, 4, 5)$, and
       check which one agrees with indexing the original array.

4. [code] **Loader throughput.** The time of one pass over the training
   set depends on the minibatch size and on how many processes prepare the
   minibatches.
    1. Time one full pass of `train_dataloader` for `FashionMNIST`
       constructed with `batch_size` of 1, 16, 64, 256, and 1024, and plot
       the throughput in images per second against `batch_size`. Explain
       the shape of the curve in terms of per-batch overhead and per-image
       work.
    1. `get_dataloader` passes the `num_workers` attribute, which
       `DataModule` sets to 4 by default, to `torch.utils.data.DataLoader`.
       For `data = FashionMNIST()`, time one pass of
       `data.train_dataloader()` with `data.num_workers` set to 0, 1, 2, 4,
       and 8, where 0 loads in the main process. State under what
       conditions more workers stop helping.

5. [code] **Upsampling and linear models.** ● Treat the resizing from
   $28 \times 28$ to $32 \times 32$ as a linear map
   $\mathbf{x}' = \mathbf{A}\mathbf{x}$ from the 784 original pixels to the
   1024 resized ones, where the columns of
   $\mathbf{A} \in \mathbb{R}^{1024 \times 784}$ are linearly independent.
    1. Show that softmax regression on $\mathbf{x}'$ can represent exactly
       the classifiers that softmax regression on $\mathbf{x}$ can.
    1. After that section, train `SoftmaxRegressionScratch`
       on data from `FashionMNIST()` and from
       `FashionMNIST(resize=(32, 32))`, with `num_inputs` set to the number
       of pixels per image. Compare the validation accuracies and explain
       any difference in light of the previous part.

[Discussions](https://d2l.discourse.group/t/49)